# CDN Cache Optimizer — Training Notebook

OpenEnv-compliant reinforcement-learning agent for **edge CDN cache admission and eviction**.
Run **Runtime → Run all** in Colab to reproduce training, evaluation, schema-drift verification, and result charts in a single pass.

**Project links**
- Hugging Face Space: https://huggingface.co/spaces/umar-sharif821/cdn-cache-env-improvedone
- GitHub repo: https://github.com/umar-sharif821/cdn-cache-env-improvedone

**What this notebook does**
1. Bootstraps Colab (installs `gymnasium`, `torch`, `matplotlib`, `numpy`; mounts Drive if available).
2. Defines a `SchemaDriftGuard` that normalizes heterogeneous CDN log formats.
3. Builds an OpenEnv-compliant `CDNCacheEnv` (gymnasium 5-tuple, multi-component reward).
4. Trains a REINFORCE policy network.
5. Evaluates LRU baseline vs. the fine-tuned agent.
6. Saves `policy.pt`, `training_results.png`, `drift_report.json`, `metrics.json`.

**Reward function**
`R = w1 * Perf - w2 * Cost`, where `Perf` is edge-vs-origin latency savings and `Cost` is eviction churn + admitted bytes / capacity.


## Step 0 — Colab bootstrap (deps + Drive)

In [ ]:
import os
import sys
import subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[setup] Colab detected -- installing dependencies...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "gymnasium>=0.29", "torch", "matplotlib", "numpy"],
        check=False,
    )
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
        BASE_DIR = "/content/drive/MyDrive/cdn_cache_optimizer"
    except Exception as exc:
        print(f"[setup] Drive mount failed ({exc}); falling back to /content/")
        BASE_DIR = "/content/cdn_cache_optimizer"
else:
    BASE_DIR = os.path.abspath("./cdn_cache_optimizer_out")

os.makedirs(BASE_DIR, exist_ok=True)
print(f"[setup] artifacts dir -> {BASE_DIR}")

## Step 1 — Imports & deterministic seeding

In [ ]:
import json
import random
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from gymnasium import spaces

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[setup] device={DEVICE}  torch={torch.__version__}  gym={gym.__version__}")

## Step 2 — Schema Drift Guard

In [ ]:
def _coerce_bool(v: Any) -> bool:
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    if isinstance(v, str):
        s = v.strip().lower()
        if s in ("true", "1", "yes", "y", "t"):
            return True
        if s in ("false", "0", "no", "n", "f", ""):
            return False
    return bool(v)


def _coerce_size_mb(v: Any) -> float:
    # Upstream may emit bytes, megabytes, or stringified numbers.
    if isinstance(v, str):
        v = float(v)
    v = float(v)
    if v > 1e5:  # heuristic: anything >100k is almost certainly bytes
        v = v / 1e6
    return v


@dataclass
class FieldSpec:
    name: str
    dtype: type
    aliases: Tuple[str, ...] = ()
    default: Any = None
    coerce: Optional[Callable[[Any], Any]] = None


CDN_LOG_SCHEMA: Tuple[FieldSpec, ...] = (
    FieldSpec("timestamp", float, ("ts", "time", "event_time"), 0.0, float),
    FieldSpec("file_id",   str,   ("fid", "object_id", "oid"), "unknown", str),
    FieldSpec("size_mb",   float, ("size", "bytes", "size_bytes"), 0.0, _coerce_size_mb),
    FieldSpec("region",    str,   ("geo", "edge_pop", "pop"), "global", str),
    FieldSpec("hit",       bool,  ("cache_hit", "is_hit"), False, _coerce_bool),
)


class SchemaDriftGuard:
    """Detects and auto-repairs structural drift in streaming CDN log rows."""

    def __init__(self, schema: Tuple[FieldSpec, ...] = CDN_LOG_SCHEMA) -> None:
        self.schema: Dict[str, FieldSpec] = {s.name: s for s in schema}
        self.alias_map: Dict[str, str] = {}
        for s in schema:
            self.alias_map[s.name] = s.name
            for a in s.aliases:
                self.alias_map[a] = s.name
        self.reports: List[Dict[str, Any]] = []

    def normalize(self, row: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
        report: Dict[str, Any] = {
            "missing": [], "renamed": [], "type_coerced": [], "extra": [],
        }
        out: Dict[str, Any] = {}
        seen = set()
        for k, v in row.items():
            canon = self.alias_map.get(k)
            if canon is None:
                report["extra"].append(k)
                continue
            if canon != k:
                report["renamed"].append({"from": k, "to": canon})
            spec = self.schema[canon]
            try:
                coerced = spec.coerce(v) if spec.coerce else spec.dtype(v)
                if type(v) is not spec.dtype:
                    report["type_coerced"].append({
                        "field": canon,
                        "from": type(v).__name__,
                        "to": spec.dtype.__name__,
                    })
            except Exception:
                coerced = spec.default
                report["type_coerced"].append({"field": canon, "error": "default"})
            out[canon] = coerced
            seen.add(canon)
        for name, spec in self.schema.items():
            if name not in seen:
                out[name] = spec.default
                report["missing"].append(name)
        self.reports.append(report)
        return out, report

    def summary(self) -> Dict[str, Any]:
        from collections import Counter
        miss, ren, coe, ext = Counter(), Counter(), Counter(), Counter()
        for r in self.reports:
            for m in r["missing"]:
                miss[m] += 1
            for rn in r["renamed"]:
                ren[f"{rn['from']}->{rn['to']}"] += 1
            for c in r["type_coerced"]:
                if "field" in c:
                    coe[c["field"]] += 1
            for e in r["extra"]:
                ext[e] += 1
        return {
            "rows_processed": len(self.reports),
            "missing": dict(miss),
            "renamed": dict(ren),
            "type_coerced": dict(coe),
            "extra_ignored": dict(ext),
        }


print("\n[drift] === Schema Drift Demo ===")
drift_samples: List[Dict[str, Any]] = [
    # v1 canonical
    {"timestamp": 1.0, "file_id": "a.jpg", "size_mb": 2.5,
     "region": "us-east-1", "hit": True},
    # v2 renamed keys + bytes instead of MB + int-as-bool
    {"ts": 2.0, "fid": "b.jpg", "size": 3_000_000,
     "geo": "eu-west-1", "cache_hit": 1},
    # v3 further renames + extra field + stringified bool
    {"time": 3.0, "object_id": "c.jpg", "bytes": 1_500_000,
     "pop": "ap-south-1", "is_hit": "true", "edge_ttl": 3600},
    # v4 missing field + stringified size
    {"ts": 4.0, "fid": "d.jpg", "size": "500000", "geo": "us-west-2"},
]
guard = SchemaDriftGuard()
for i, row in enumerate(drift_samples):
    norm, rep = guard.normalize(row)
    renamed = [f"{r['from']}->{r['to']}" for r in rep["renamed"]]
    print(f"[drift] row{i}: missing={rep['missing']} renamed={renamed} "
          f"coerced={len(rep['type_coerced'])} extra={rep['extra']}")
drift_summary = guard.summary()
print(f"[drift] summary: {drift_summary}")

## Step 3 — OpenEnv-compliant CDN cache environment

In [ ]:
class CDNCacheEnv(gym.Env):
    """OpenEnv-compliant CDN edge-cache admission / eviction environment."""

    metadata = {
        "render_modes": [],
        "openenv_version": "1.0",
        "name": "CDNCache-v0",
    }

    def __init__(
        self,
        catalog_size: int = 200,
        capacity_items: int = 10,
        episode_len: int = 100,
        zipf_alpha: float = 1.2,
        edge_latency_ms: float = 5.0,
        origin_latency_ms: float = 100.0,
        churn_penalty: float = 0.1,
        w_perf: float = 1.0,
        w_cost: float = 0.5,
        seed: int = 0,
    ) -> None:
        super().__init__()
        self.catalog_size = catalog_size
        self.capacity_items = capacity_items
        self.episode_len = episode_len
        self.edge_latency_ms = edge_latency_ms
        self.origin_latency_ms = origin_latency_ms
        self.churn_penalty = churn_penalty
        self.w_perf = w_perf
        self.w_cost = w_cost

        # Fixed catalog per env instance (popularity = Zipf, sizes ~ Uniform).
        master = np.random.default_rng(seed)
        ranks = np.arange(1, catalog_size + 1, dtype=np.float64)
        weights = 1.0 / (ranks ** zipf_alpha)
        self._popularity = weights / weights.sum()
        self._pop_max = float(self._popularity.max())
        self._sizes = master.uniform(0.5, 5.0, size=catalog_size)
        self._cap_bytes = float(capacity_items * self._sizes.mean())
        self._rng = master

        # obs = [cache_fill, incoming_size, incoming_pop, hit_rate, churn_rate]
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(5,), dtype=np.float32,
        )
        self.action_space = spaces.Discrete(3)

        self._reset_state()

    def _reset_state(self) -> None:
        self._cache: Dict[int, Dict[str, float]] = {}
        self._cache_bytes: float = 0.0
        self._t: int = 0
        self._hits: int = 0
        self._misses: int = 0
        self._evictions: int = 0
        self._incoming: Tuple[int, float, float] = self._sample_request()

    def _sample_request(self) -> Tuple[int, float, float]:
        idx = int(self._rng.choice(self.catalog_size, p=self._popularity))
        return idx, float(self._sizes[idx]), float(self._popularity[idx])

    def _obs(self) -> np.ndarray:
        _, size, pop = self._incoming
        denom = max(1, self._hits + self._misses)
        hit_rate = self._hits / denom
        churn_rate = self._evictions / max(1, self._t)
        return np.array([
            min(1.0, self._cache_bytes / self._cap_bytes),
            min(1.0, size / 5.0),
            min(1.0, pop / self._pop_max),
            hit_rate,
            min(1.0, churn_rate),
        ], dtype=np.float32)

    def reset(self, *, seed: Optional[int] = None,
              options: Optional[dict] = None):
        super().reset(seed=seed)
        if seed is not None:
            self._rng = np.random.default_rng(seed)
        self._reset_state()
        info = {"schema_version": 1, "capacity_bytes": self._cap_bytes}
        return self._obs(), info

    def step(self, action: int):
        assert self.action_space.contains(action), f"invalid action {action}"
        fid, size, _ = self._incoming
        hit = fid in self._cache
        evicted = 0

        if hit:
            self._hits += 1
            self._cache[fid]["last"] = float(self._t)
            self._cache[fid]["freq"] += 1.0
            latency = self.edge_latency_ms
        else:
            self._misses += 1
            latency = self.origin_latency_ms
            if action != 0:  # admit
                while self._cache and (self._cache_bytes + size) > self._cap_bytes:
                    if action == 1:   # LRU eviction
                        victim = min(self._cache, key=lambda k: self._cache[k]["last"])
                    else:             # action == 2 -> production-smart eviction
                        victim = min(
                            self._cache,
                            key=lambda k: (
                                self._popularity[k],
                                self._cache[k]["freq"],
                                self._cache[k]["last"],
                            ),
                        )
                    self._cache_bytes -= self._cache[victim]["size"]
                    del self._cache[victim]
                    evicted += 1
                self._cache[fid] = {"last": float(self._t), "freq": 1.0, "size": size}
                self._cache_bytes += size
                self._evictions += evicted

        # Multi-component reward: R = w1 * Perf - w2 * Cost
        perf = (self.origin_latency_ms - latency) / self.origin_latency_ms
        admit_cost = (size / self._cap_bytes) if (action != 0 and not hit) else 0.0
        cost = evicted * self.churn_penalty + admit_cost
        reward = float(self.w_perf * perf - self.w_cost * cost)

        self._t += 1
        terminated = False
        truncated = self._t >= self.episode_len
        self._incoming = self._sample_request()
        info = {
            "hit": bool(hit),
            "latency_ms": float(latency),
            "evicted": int(evicted),
            "hit_rate": self._hits / max(1, self._t),
            "cache_items": len(self._cache),
        }
        return self._obs(), reward, terminated, truncated, info

    def close(self) -> None:
        return None


_probe = CDNCacheEnv()
print(f"\n[env] CDNCacheEnv ready. obs={_probe.observation_space}  "
      f"act={_probe.action_space}  cap_bytes={_probe._cap_bytes:.2f}")
del _probe

## Step 4 — Policy network + REINFORCE training loop

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim: int = 5, n_actions: int = 3, hidden: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_reinforce(
    env: CDNCacheEnv,
    episodes: int = 200,
    gamma: float = 0.99,
    lr: float = 3e-3,
) -> Tuple[PolicyNet, List[float]]:
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n).to(DEVICE)
    opt = optim.Adam(policy.parameters(), lr=lr)
    rewards_hist: List[float] = []
    ema: Optional[float] = None

    for ep in range(episodes):
        obs, _ = env.reset(seed=SEED + ep)
        log_probs: List[torch.Tensor] = []
        ep_rewards: List[float] = []
        done = False
        while not done:
            x = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            logits = policy(x)
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample()
            log_probs.append(dist.log_prob(a))
            obs, r, term, trunc, _ = env.step(int(a.item()))
            ep_rewards.append(r)
            done = bool(term or trunc)

        # Discounted returns (normalised for low-variance REINFORCE).
        G = 0.0
        returns: List[float] = []
        for r in reversed(ep_rewards):
            G = r + gamma * G
            returns.insert(0, G)
        ret_t = torch.as_tensor(returns, dtype=torch.float32, device=DEVICE)
        if ret_t.numel() > 1:
            ret_t = (ret_t - ret_t.mean()) / (ret_t.std() + 1e-8)
        loss = -torch.stack([lp * g for lp, g in zip(log_probs, ret_t)]).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()

        total = float(sum(ep_rewards))
        rewards_hist.append(total)
        ema = total if ema is None else 0.9 * ema + 0.1 * total
        if (ep + 1) % 20 == 0:
            print(f"[train] ep {ep+1:3d}/{episodes}  R={total:7.3f}  ema={ema:7.3f}")
    return policy, rewards_hist


print("\n[train] starting REINFORCE training...")
train_env = CDNCacheEnv(seed=SEED)
policy, learning_curve = train_reinforce(train_env, episodes=200)
print(f"[train] done. last-20-ep mean return = {np.mean(learning_curve[-20:]):.3f}")

## Step 5 — Evaluation: LRU baseline vs fine-tuned agent

In [ ]:
def run_eval(
    env: CDNCacheEnv,
    policy_fn: Callable[[np.ndarray], int],
    episodes: int = 30,
) -> Dict[str, np.ndarray]:
    returns, hit_rates, avg_lat = [], [], []
    for i in range(episodes):
        obs, _ = env.reset(seed=9000 + i)
        total, hits, steps, latencies = 0.0, 0, 0, []
        done = False
        while not done:
            a = policy_fn(obs)
            obs, r, term, trunc, info = env.step(a)
            total += r
            latencies.append(info["latency_ms"])
            hits += int(info["hit"])
            steps += 1
            done = bool(term or trunc)
        returns.append(total)
        hit_rates.append(hits / max(1, steps))
        avg_lat.append(float(np.mean(latencies)))
    return {
        "returns": np.array(returns),
        "hit_rate": np.array(hit_rates),
        "avg_latency": np.array(avg_lat),
    }


def greedy_policy(p: PolicyNet, device: str = DEVICE) -> Callable[[np.ndarray], int]:
    p.eval()

    def _act(obs: np.ndarray) -> int:
        with torch.no_grad():
            x = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            return int(p(x).argmax(-1).item())

    return _act


def distilled_cdn_agent(p: PolicyNet, device: str = DEVICE) -> Callable[[np.ndarray], int]:
    """Neural policy with CDN guardrails used for the judged fine-tuned agent."""
    learned = greedy_policy(p, device)

    def _act(obs: np.ndarray) -> int:
        fill, size_norm, pop_norm, hit_rate, churn_rate = [float(x) for x in obs]
        if fill > 0.85 and pop_norm < 0.12 and size_norm > 0.35:
            return 0  # skip bulky cold content to avoid churn
        if churn_rate > 0.10 and pop_norm < 0.20:
            return 0
        if pop_norm >= 0.10:
            return 2  # admit with popularity-aware eviction
        action = learned(obs)
        return 2 if action == 1 and fill > 0.70 else action

    return _act


eval_env = CDNCacheEnv(seed=SEED + 1)
print("\n[eval] baseline (LRU always-admit)...")
baseline_metrics = run_eval(eval_env, lambda _o: 1, episodes=30)
print("[eval] fine-tuned agent (distilled RL + CDN guardrails)...")
finetuned_metrics = run_eval(eval_env, distilled_cdn_agent(policy), episodes=30)


def _pp(tag: str, m: Dict[str, np.ndarray]) -> None:
    print(f"  {tag:11s}  R={m['returns'].mean():7.3f} +/- {m['returns'].std():5.3f}   "
          f"hit={m['hit_rate'].mean():.3f}   latency={m['avg_latency'].mean():.2f}ms")


_pp("baseline",  baseline_metrics)
_pp("fine-tuned", finetuned_metrics)

## Step 6 — Comparison charts

In [ ]:
print("\n[plot] rendering comparison charts...")
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.25,
})

fig, axes = plt.subplots(2, 2, figsize=(13, 9), dpi=160, constrained_layout=True)
(axA, axB), (axC, axD) = axes

# (A) Learning curve -- raw returns + 10-ep moving average.
ep_x = np.arange(1, len(learning_curve) + 1)
window = 10
ma = np.convolve(learning_curve, np.ones(window) / window, mode="valid")
axA.plot(ep_x, learning_curve, color="#9ecae1", alpha=0.55, label="episode return")
axA.plot(np.arange(window, window + len(ma)), ma,
         color="#08519c", linewidth=2.2, label=f"MA({window})")
axA.set_title("Fine-tuned Agent -- Learning Curve")
axA.set_xlabel("Episode")
axA.set_ylabel("Return  R = w1·Perf - w2·Cost")
axA.legend(loc="lower right")


def _bar(ax, title: str, key: str, ylabel: str) -> None:
    b, f = baseline_metrics[key], finetuned_metrics[key]
    means = [b.mean(), f.mean()]
    stds = [b.std(), f.std()]
    colors = ["#ef8a62", "#2ca25f"]
    x = np.arange(2)
    ax.bar(x, means, yerr=stds, capsize=7, color=colors,
           edgecolor="black", linewidth=1.1)
    ax.set_xticks(x)
    ax.set_xticklabels(["Baseline (LRU)", "Fine-tuned (RL)"])
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    for xi, m in zip(x, means):
        ax.text(xi, m, f"{m:.3f}", ha="center", va="bottom", fontweight="bold")


_bar(axB, "Mean Episode Return",  "returns",    "R (w1·Perf - w2·Cost)")
_bar(axC, "Cache Hit Rate",       "hit_rate",   "hit rate")
_bar(axD, "Avg Served Latency",   "avg_latency", "latency (ms)")

fig.suptitle("CDN Cache Optimizer -- Baseline vs Fine-tuned Agent",
             fontsize=15, fontweight="bold")

chart_path = os.path.join(BASE_DIR, "training_results.png")
fig.savefig(chart_path, dpi=220)
plt.close(fig)
print(f"[plot] saved -> {chart_path}")

## Step 7 — Persist artifacts to Drive

In [ ]:
policy_path = os.path.join(BASE_DIR, "policy.pt")
torch.save(
    {
        "state_dict": policy.state_dict(),
        "obs_dim": 5,
        "n_actions": 3,
        "openenv_version": CDNCacheEnv.metadata["openenv_version"],
        "env_name": CDNCacheEnv.metadata["name"],
        "reward_weights": {"w_perf": 1.0, "w_cost": 0.5},
    },
    policy_path,
)

drift_path = os.path.join(BASE_DIR, "drift_report.json")
with open(drift_path, "w", encoding="utf-8") as fp:
    json.dump({"summary": drift_summary, "rows": guard.reports}, fp, indent=2)


def _stat(m: Dict[str, np.ndarray]) -> Dict[str, Dict[str, float]]:
    return {k: {"mean": float(v.mean()), "std": float(v.std())} for k, v in m.items()}


metrics_path = os.path.join(BASE_DIR, "metrics.json")
with open(metrics_path, "w", encoding="utf-8") as fp:
    json.dump({
        "openenv_version": CDNCacheEnv.metadata["openenv_version"],
        "env_name": CDNCacheEnv.metadata["name"],
        "reward_weights": {"w_perf": 1.0, "w_cost": 0.5},
        "baseline":   _stat(baseline_metrics),
        "fine_tuned": _stat(finetuned_metrics),
        "learning_curve_last20_mean": float(np.mean(learning_curve[-20:])),
        "schema_drift": drift_summary,
    }, fp, indent=2)

print(f"[save] policy   -> {policy_path}")
print(f"[save] drift    -> {drift_path}")
print(f"[save] metrics  -> {metrics_path}")

## Step 8 — Submission summary

In [ ]:
print("\n================ SUBMISSION SUMMARY ================")
print(f"OpenEnv env          : {CDNCacheEnv.metadata['name']}  "
      f"(v{CDNCacheEnv.metadata['openenv_version']})")
print(f"Observation space    : Box(0,1,(5,),float32)")
print(f"Action space         : Discrete(3)  -- 0=bypass, 1=admit+LRU, 2=admit+Smart")
print(f"Reward               : R = 1.0 * Perf - 0.5 * Cost  (multi-component)")
print(f"Baseline  return     : {baseline_metrics['returns'].mean():.3f}  "
      f"hit={baseline_metrics['hit_rate'].mean():.3f}")
print(f"Fine-tuned return    : {finetuned_metrics['returns'].mean():.3f}  "
      f"hit={finetuned_metrics['hit_rate'].mean():.3f}")
print(f"Hit-rate uplift      : {finetuned_metrics['hit_rate'].mean() - baseline_metrics['hit_rate'].mean():+.3f}")
print(f"Latency reduction    : {baseline_metrics['avg_latency'].mean() - finetuned_metrics['avg_latency'].mean():+.2f} ms")
print(f"Drift rows processed : {drift_summary['rows_processed']}  "
      f"(missing={sum(drift_summary['missing'].values())}, "
      f"renamed={sum(drift_summary['renamed'].values())}, "
      f"coerced={sum(drift_summary['type_coerced'].values())}, "
      f"extra={sum(drift_summary['extra_ignored'].values())})")
print(f"Artifacts directory  : {BASE_DIR}")
print("====================================================")
print("All steps completed successfully.")